In [2]:
import os
import sys
import glob
import csv
import random
import logging
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Literal, Union
from pathlib import Path
from functools import partial
from collections import Counter
from instanovo.utils.data_handler import SpectrumDataFrame

from instanovo.transformer.dataset import remove_modifications as clean_peptide

# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

[04/16/25 01:36:00] INFO     Enabling RDKit 2024.09.6 jupyter extensions                             ]8;id=83295;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py\__init__.py]8;;\:]8;id=801822;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py#22\22]8;;\

In [3]:
from common.utils import collect_files, get_or_create_folder, load_ipc_files
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_PROCESSED_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    ROOT_DIR,
    BASE_REPORTS_CSV_DIR,
    IDENTITY_FILE_PATHS, 
    BLACKLIST_FILE_PATHS,
)

In [4]:
logger_config = get_logger_config(subdir="scripts")
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)

In [5]:
# Collect each unique_peptide.csv file
peptides_file_paths = [
    path
    for path in collect_files(BASE_REPORTS_CSV_DIR, ext="csv")
    if "unique_peptides" in path
]
projects_dirs = glob.glob(f"{BASE_RAW_DATA_DIR}/*/")
projects_names = [project_dir.split("/")[-2] for project_dir in projects_dirs]
project_dirs_to_ignore = ["PXD044641_PXD035158"]  #

assert projects_dirs, projects_dirs
assert peptides_file_paths, peptides_file_paths

In [15]:
df = pd.concat([pd.read_csv(file) for file in peptides_file_paths], ignore_index=True)
df.head(20)

,Unique Peptides
0,HNGTGGR
1,SQNCHNSSSR
2,AAGMNHTK
3,ANASHDQPQK
4,HNDSGASECR
5,GGGGGGGGGGGGGSGSSSGSSTSR
6,RQQQQQQQQQQQQK
7,QQQQQQQQQQQQK
8,KNDSGAYR
9,KCLNHTTQK


In [5]:
df["Unique Peptides"].describe()

NameError: name 'df' is not defined

## Split without Kevin constraint

In [17]:
unique_peptides_df = df["Unique Peptides"].drop_duplicates()

In [18]:
indices = np.arange(len(unique_peptides_df))
np.random.shuffle(indices)
split_ratio = 0.8

split_seperator = int(len(unique_peptides_df) * split_ratio)

# Shuffle the DataFrame indices
shuffled_df = unique_peptides_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train/test split
train_peptides_df = shuffled_df.iloc[:split_seperator].reset_index(drop=True)
test_peptides_df = shuffled_df.iloc[split_seperator:].reset_index(drop=True)

In [19]:
assert len(train_peptides_df) == 35980, len(train_peptides_df)
assert len(test_peptides_df) == 8996, len(test_peptides_df)

In [20]:
def write_split(
    source_dir: Path | str,
    project_name: Path | str,
    split_name: Literal["train", "valid", "test"],  # noqa
    algorithm_version: Literal["v0", "v1", "v2", "v2.1"],
    potential_peptides_set: set,
    max_charge: int = 10,
    drop_unmodified: bool = False,
):
    file_paths = collect_files(location=source_dir)

    sdf, _ = load_ipc_files(file_paths)
    logger.info(f"Loaded {len(sdf)} entries from {source_dir}")

    # Filter by charge and peptide set
    sdf = sdf[
        (sdf["precursor_charge"] <= max_charge)
        & (sdf["precursor_charge"] > 0)
        & (sdf["peptide"].apply(lambda x: clean_peptide(x) in potential_peptides_set))
    ]
    logger.info(f"Got {len(sdf)} spectra after filtering by precursor charge")
    logger.info(f"Starting {split_name} split for project {project_name}")

    # Identify missing and fake modifications
    is_missing = sdf["modified_peptide"].isna()
    is_fake = sdf["modified_peptide"] == sdf["peptide"]

    logger.info(f"Found {is_missing.sum()} rows with missing modified_peptide")
    logger.info(
        f"Found {is_fake.sum()} rows with fake modified_peptide (same as peptide)"
    )

    # Treat fake modifications as unmodified
    is_unmodified = is_missing | is_fake

    if drop_unmodified:
        logger.info("Filtering out rows with missing or fake modified_peptide")
        sdf = sdf[~is_unmodified]
        logger.info(f"Left with {len(sdf)} rows after dropping unmodified rows")
    else:
        logger.info("Filling missing modified_peptide with related peptide")
        sdf.loc[is_missing, "modified_peptide"] = sdf["peptide"]

    assert (
        sdf["precursor_charge"].between(1, max_charge).all()
    ), "Some precursor_charge values are out of range."
    assert all(
        clean_peptide(p) in potential_peptides_set for p in sdf["peptide"]
    ), "Some peptides are not in the allowed set."
    assert (
        sdf["modified_peptide"].isna().sum() == 0
    ), "Every row should have modified_peptide set"

    # Save final file
    target_path = BASE_PROCESSED_DATA_DIR / project_name
    filename = f"dataset-ms-glyco_{algorithm_version}_{split_name}.parquet"
    sdf.to_parquet(path=target_path / filename, index=False)
    logger.info(
        f"Saved {len(sdf)} spectra for {split_name} to {target_path} for project {project_name}"
    )

In [21]:
logger.info("Starting to split the dataset but using random split")

# Version 0 for train/test split: Constraint free split
# NOTE: This code is broken because of the param drop_unmodified
algorithm_version = "v0"
project_dirs_to_ignore = ["PXD044641_PXD035158"]  #
# DOCME: Replace the [] by projects_dirs to make the to script run
for project_dir in []:  # projects_dirs:
    project_name = project_dir.split("/")[-2]
    if project_name in project_dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )
    for split_name, peptide_set in [
        ("train", set(train_peptides_df)),
        # ("val", set(val_peptides_df)),
        ("test", set(test_peptides_df)),
    ]:
        write_split(
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-15 15:14:17,590 - __main__ - INFO - Starting to split the dataset but using random split


In [22]:
kevin_train_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "train_blacklist_overlap_identity_splits_massivekb_from_kevin_1067866_with_glyco_projects_44976_found_15499.csv"
)["Overlapped train peptides"].unique()
kevin_test_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "test_overlap_identity_splits_massivekb_from_kevin_33575_with_glyco_projects_44976_found_4136.csv"
)["Overlapped test peptides"].unique()
kevin_val_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "valid_overlap_identity_splits_massivekb_from_kevin_13062_with_glyco_projects_44976_found_495.csv"
)["Overlapped valid peptides"].unique()

In [23]:
# Version 1 for train/test/valid split => peptide is used as fallback for modified_peptide
logger.info("Starting to split the dataset but taking into account kevin's suggestion")
project_dirs_to_ignore = ["PXD044641_PXD035158"]  #
# Focus on massivekb

algorithm_version = "v1"
# TODO: Uncomment the # projects_dirs to run the script
for project_dir in []:  # projects_dirs:
    project_name = project_dir.split("/")[-2]

    if project_name in project_dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )

    for split_name, kevin_peptide_set in [
        ("train", set(kevin_train_peptides_array)),
        ("valid", set(kevin_val_peptides_array)),
        ("test", set(kevin_test_peptides_array)),
    ]:
        write_split(
            drop_unmodified=False,
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(kevin_peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-15 15:14:17,803 - __main__ - INFO - Starting to split the dataset but taking into account kevin's suggestion


### Dataset Split Algo Version 2/2.1

In [ ]:
# Version 2 or Version 2.1 for train/test/valid split => All rows with missing modified_peptides are filtered out. But is version 2.1 we also filter out fake modifications defined as modifications for which modified_peptide is equal to peptide.
logger.info("Starting to split the dataset but taking into account kevin's suggestion")
project_dirs_to_ignore = ["PXD044641_PXD035158"]  #

# Focus on massivekb
algorithm_version = "v2.1"  # Version 2.1
for project_dir in projects_dirs:  # projects_dirs:
    project_name = project_dir.split("/")[-2]

    if project_name in project_dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )

    for split_name, kevin_peptide_set in [
        ("train", set(kevin_train_peptides_array)),
        ("valid", set(kevin_val_peptides_array)),
        ("test", set(kevin_test_peptides_array)),
    ]:
        write_split(
            # The difference here
            drop_unmodified=True,
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(kevin_peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-15 15:14:17,864 - __main__ - INFO - Starting to split the dataset but taking into account kevin's suggestion
2025-04-15 15:14:17,866 - __main__ - INFO - Collected 27 of project PXD026629 files from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/
Processing files:  44%|████▍     | 12/27 [00:06<00:10,  1.38it/s]2025-04-15 15:14:24,417 - common.utils - INFO - Processing file 12: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/YangLuJie-LPS4h-3.ipc


### Dataset Split Algo Version 3

1. Find a list of all unique unmodified peptides in all the glyco data — Let's call it set **G**
2. Let my data splits be **E**, basically the train/valid/test splits I provided for all the datasets
3. Find the intersection **G ∩ E**, use the existing labels for those
4. Find the unlabeled set **G - E**, and manually label these  
    a. Use the intersection **G ∩ E** to find the number of peptides per data split already labelled  
    b. Randomly distribute **G - E** such that you get roughly an 80:10:10 split for train:valid:test once you combine all your data splits  
    c. The combination of **G ∪ E** and **G - E** should give you your final data splits for all data

In [30]:

# All the peptides in the glyco dataset
glyco_peptides_df, _ = load_ipc_files([BASE_REPORTS_CSV_DIR / project_name / "unique_peptides.csv" for project_name in projects_names if project_name not in project_dirs_to_ignore], format="csv")
glyco_peptides_set = set(glyco_peptides_df) 


Processing files:   0%|          | 0/8 [00:00<?, ?it/s]2025-04-16 02:23:09,268 - common.utils - INFO - Processing file 0: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/PXD026629/unique_peptides.csv
2025-04-16 02:23:09,285 - common.utils - INFO - Processing file 1: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/PXD031032/unique_peptides.csv
2025-04-16 02:23:09,302 - common.utils - INFO - Processing file 2: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/PXD031025/unique_peptides.csv
2025-04-16 02:23:09,307 - common.utils - INFO - Processing file 3: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/PXD026649/unique_peptides.csv
2025-04-16 02:23:09,321 - common.utils - INFO - Processing file 4: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/reports/csv_misc/PXD047898/unique_peptides.csv
2025-04-16 02:23:09,327 - common.utils - INFO - Processing file 5: /home/hj

In [31]:
# Here, we want to merge all the identity files and use them to do the first splits
kevin_merged_splits_df, kevin_merged_splits_summary = load_ipc_files(IDENTITY_FILE_PATHS, format="csv")

Processing files: 100%|██████████| 3/3 [00:03<00:00,  1.16s/it]
2025-04-16 02:23:16,569 - common.utils - INFO - Files loading completed.


In [32]:
len(kevin_merged_splits_df)

2435413

In [33]:
tuple_counts = Counter(kevin_merged_splits_df[["sequence", "split"]].itertuples(index=False, name=None))
duplicated_tuples = [t for t, count in tuple_counts.items() if count > 1]
print(f"Number of unique tuples that have duplicates: {len(duplicated_tuples)}")


Number of unique tuples that have duplicates: 490007


In [34]:
# Specify the output file path.
# Those duplicates come from massivekb, actually.
duplicates_tuples_output_file = ROOT_DIR / '.trash_local/duplicated_identity_files_tuples.csv' 

# Check if the file already exists
if not os.path.exists(duplicates_tuples_output_file):
    # Write the duplicated_tuples to a CSV file
    with open(duplicates_tuples_output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['sequence', 'split'])  # Write header
        writer.writerows(duplicated_tuples)  # Write the tuples

    print(f"Duplicated tuples have been written to {duplicates_tuples_output_file}")
else:
    print(f"File {duplicates_tuples_output_file} already exists. Skipping write operation.")

File /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/.trash_local/duplicated_identity_files_tuples.csv already exists. Skipping write operation.


In [35]:
# Drop duplicate
kevin_merged_splits_df = kevin_merged_splits_df.drop_duplicates()


In [36]:
print("After dropping duplicates, the number of rows is: ", len(kevin_merged_splits_df))

After dropping duplicates, the number of rows is:  1639892


In [40]:
split_counts = kevin_merged_splits_df.groupby('sequence')['split'].nunique()
split_counts.to_csv(BASE_REPORTS_CSV_DIR / "merged_identity_splits_files_peptides_with_many_split_values.csv", index=False)

In [41]:
multi_split_sequences = split_counts[split_counts > 1]
print(f"Sequences with multiple splits: {len(multi_split_sequences)}")
print(multi_split_sequences.head())

Sequences with multiple splits: 2557
sequence
AAAAECDVVMAATEPELLDDQEAK        2
AAAAECDVVMAATEPELLDDQEAKR       2
AAAAGPGAALSPRPCDSDPATPGAQSPK    2
AAAAPDSRVSEEENLK                2
AAAMTPPEEELK                    2
Name: split, dtype: int64


In [42]:
# Keep only the first occurrence of each sequence
kevin_merged_splits_df = kevin_merged_splits_df.drop_duplicates(subset='sequence', keep='first')
print("After dropping duplicates, the number of rows is: ", len(kevin_merged_splits_df))
# Recalculate multi_split_sequences to confirm it's empty
split_counts = kevin_merged_splits_df.groupby('sequence')['split'].nunique()
multi_split_sequences = split_counts[split_counts > 1]

print(f"Sequences with multiple splits after processing: {len(multi_split_sequences)}")

After dropping duplicates, the number of rows is:  1637335
Sequences with multiple splits after processing: 0


In [43]:
kevin_merged_splits_df.to_csv(
    BASE_REPORTS_CSV_DIR / "merged_identity_splits_files_peptides_with_unique_split_value.csv", index=False
)

In [44]:
multi_split_sequences

Series([], Name: split, dtype: int64)

In [48]:
# Since there is no duplicate in kevin_merged_splits_set, the only pros of defining kevin_merged_splits_set 
# here is to leverage the usage of the memory to speed-up operations as a set uses hash table
peptide_to_split = dict(kevin_merged_splits_df[["sequence", "split"]].itertuples(index=False, name=None))
kevin_merged_peptides_set = set(peptide_to_split)

# kevin_merged_splits_set = set(kevin_merged_splits_df[["sequence", "split"]].itertuples(index=False, name=None))

In [49]:
len(kevin_merged_peptides_set)


1637335

In [ ]:
# IMPORTANT: Instead of creating train/valid/test splits for a given source_dir with only one call of 
# write_split_novel, we will implement the write_split_novel so that we will create each split with a
# separate call for write_split_novel. The avantage is to be able to run the algorithm locally without
# running out of memory.

def clean_peptide_filter(peptide: str, existing_splits: dict[str, str], blacklist:list | None=None)->bool:
    blacklist = [] if blacklist is None else blacklist
    cleaned_peptide = clean_peptide(peptide)
    return (existing_splits.get(cleaned_peptide) == split_name) and (cleaned_peptide not in blacklist)
    


def write_split_novel(
    source_dir: Union[Path, str],
    project_name: Union[Path, str],
    split_name: Literal["train", "valid", "test"],
    algorithm_version: Literal["v3"],
    existing_splits: dict[str, str],
    all_projects_dirs: list[Path],
    blacklist: list | None = None,
    max_charge: int = 10,
):
    """New split function with novel peptide allocation logic."""
    entries_count = 0
    
    local_cleaned_peptide_filter  = partial(
        clean_peptide_filter, existing_splits=existing_splits, blacklist=blacklist
    )
    
    # 2. Calculate E and G-E
    E = existing_splits["train"].union(existing_splits["valid"], existing_splits["test"])
    G_minus_E = list(G - E)
    random.shuffle(G_minus_E)

    # 3. Allocate G-E peptides to splits
    split_allocation = {
        "train": int(0.8 * len(G_minus_E)),
        "valid": int(0.1 * len(G_minus_E)),
        "test": len(G_minus_E) - int(0.9 * len(G_minus_E))
    }
    
    allocated = {
        "train": set(G_minus_E[:split_allocation["train"]]),
        "valid": set(G_minus_E[split_allocation["train"]:split_allocation["train"]+split_allocation["valid"]]),
        "test": set(G_minus_E[split_allocation["train"]+split_allocation["valid"]:])
    }

    # 4. Create combined peptide set for this split
    combined_set = (existing_splits[split_name].intersection(G)).union(allocated[split_name])

    # 5. Load and process data
    file_paths = collect_files(location=source_dir)
    sdf, _ = load_ipc_files(file_paths)
    logger.info(f"Loaded {len(sdf)} entries from {source_dir}")
    # TODO : blacklist thing
    
    # Apply filters (fixed parentheses)
    sdf = sdf[
        (sdf["precursor_charge"] <= max_charge) &
        (sdf["precursor_charge"] > 0) &
        (sdf["peptide"].apply(local_cleaned_peptide_filter))
    ]
    
    logger.info(
        f"Got {len(sdf)} entries after filtering by precursor charge and peptide belonging to {split_name} split"
    )
    
    # Handle modifications
    no_modification = sdf["modified_peptide"] == sdf["peptide"]
    logger.info(f"Found {len(no_modification)} unmodified peptides entries but already default to peptide")
    is_missing = sdf["modified_peptide"].isna()
    logger.info(f"Found {len(is_missing)} empty/missing modified_peptide entries..., defaulting their modified_peptide to peptide")
    sdf.loc[is_missing, "modified_peptide"] = sdf["peptide"]  # Fill missing
    logger.info(f"Starting {split_name} split for project {project_name}")

    
    # Save results
    target_path = BASE_PROCESSED_DATA_DIR / project_name
    filename = f"dataset-ms-glyco_{algorithm_version}_{split_name}.parquet"
    sdf.to_parquet(target_path / filename, index=False)

In [ ]:
write_split_novel(
    
)

In [ ]:
# Existing splits
existing_splits = {
    "train": set(kevin_train_peptides_array),
    "valid": set(kevin_val_peptides_array),
    "test": set(kevin_test_peptides_array)
}

# Process all projects
for project_dir in projects_dirs:
    project_name = project_dir.name
    if project_name in project_dirs_to_ignore:
        continue
        
    for split_name in ["train", "valid", "test"]:
        write_split_novel(
            source_dir=BASE_RAW_DATA_DIR / project_name,
            project_name=project_name,
            split_name=split_name,
            algorithm_version="v3",
            existing_splits=existing_splits,
            all_projects_dirs=projects_dirs,
            max_charge=10
        )

In [ ]:
# rr["modified_peptide"].head(100)

## Attempt to analyze the content of the split files

### Split Version 2.1 - Content analysis

In [ ]:
split_version = 2.1
logger.info(f"Split version {split_version} content analysis")

for split in ("train", "valid", "test"):
    dfs = []
    for project_dir in projects_dirs:  # projects_dirs:
        project_name = project_dir.split("/")[-2]

        if project_name in project_dirs_to_ignore:
            logger.info(
                f"Skipping project {project_name} as part of projects to ignore"
            )
            continue

        file_path = f"{project_name}/dataset-ms-glyco_v{split_version}_{split}.parquet"
        df = pd.read_parquet(BASE_PROCESSED_DATA_DIR / file_path)
        logger.info(f"Got {len(df)} rows from {file_path} for project {project_name}")
        dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)
    logger.info(
        f"Overall {len(result)} rows for project {project_name} {split} v{split_version}"
    )

    result[["peptide", "modified_peptide"]].to_csv(
        BASE_REPORTS_CSV_DIR
        / f"version{split_version}_{split}_split_peptide_and_modified_peptides.csv",
        index=False,
    )

In [ ]:
train_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_train_split_peptide_and_modified_peptides.csv",
)

train_peptide_and_modified_df.describe()

In [ ]:
valid_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_valid_split_peptide_and_modified_peptides.csv",
)
valid_peptide_and_modified_df.describe()

In [ ]:
test_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_test_split_peptide_and_modified_peptides.csv",
)
test_peptide_and_modified_df.describe()

### Split Version 1 - Content analysis


In [ ]:
split_version = 1
logger.info(f"Split version {split_version} content analysis")

for split in ("train", "valid", "test"):
    dfs = []
    for project_dir in projects_dirs:  # projects_dirs:
        project_name = project_dir.split("/")[-2]

        if project_name in project_dirs_to_ignore:
            logger.info(
                f"Skipping project {project_name} as part of projects to ignore"
            )
            continue
        logger.info(f"Reading {project_dir}")
        df = pd.read_parquet(
            "/home/hjisaac/AI4Science/instanovo_instadeep/glycodata_processed_version1/processed"
            f"/{project_name}/dataset-ms-glyco_v{split_version}_{split}.parquet"
        )
        dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)

    result[["peptide", "modified_peptide"]].to_csv(
        BASE_REPORTS_CSV_DIR
        / f"version{split_version}_{split}_split_peptide_and_modified_peptides.csv",
        index=False,
    )

In [ ]:
train_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_train_split_peptide_and_modified_peptides.csv",
)

train_peptide_and_modified_df.describe()

In [ ]:
valid_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_valid_split_peptide_and_modified_peptides.csv",
)
valid_peptide_and_modified_df.describe()

In [ ]:
test_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_test_split_peptide_and_modified_peptides.csv",
)
test_peptide_and_modified_df.describe()